In [123]:
import sys
sys.path.append('/content')

In [124]:
!pip install yfinance transformers torch streamlit newsapi-python


In [91]:
!mkdir -p behavioral_alpha/ingestion
!mkdir -p behavioral_alpha/nlp
!mkdir -p behavioral_alpha/indicators
!mkdir -p behavioral_alpha/signals
!mkdir -p behavioral_alpha/database
!mkdir -p behavioral_alpha/dashboard


In [92]:
%%writefile behavioral_alpha/ingestion/news_feed.py
import requests

API_KEY = "4859adcb544f4e27a15ace3d096ff803"

def fetch_news(ticker=None):
    """
    Fetch live business & stock market news from NewsAPI.
    If ticker is provided, fetch news related to that company.
    Returns a list of headlines.
    """
    try:
        query = f"&q={ticker}" if ticker else ""
        url = (
            "https://newsapi.org/v2/top-headlines?"
            "category=business&"
            "language=en"
            f"{query}&"
            f"apiKey={API_KEY}"
        )

        response = requests.get(url, timeout=10)
        data = response.json()

        if data.get("status") != "ok":
            return [f"API Error: {data.get('message', 'Unknown error')}"]

        headlines = [article.get("title") for article in data.get("articles", []) if article.get("title")]
        return headlines if headlines else ["No news found"]

    except Exception as e:
        return [f"Error fetching news: {e}"]



Overwriting behavioral_alpha/ingestion/news_feed.py


In [106]:
from behavioral_alpha.ingestion.news_feed import fetch_news

news = fetch_news()
print("Fetched Live News:")
for i, headline in enumerate(news, 1):
    print(f"{i}. {headline}")


Fetched Live News:
1. Jony Ive and Sam Altman say they finally have an AI hardware prototype - The Verge
2. Defense Stocks Fall as Trump Pushes Ukraine Peace Deal. What It Means for Markets. - Barron's
3. Michael Burry launches newsletter to lay out his AI bubble views after deregistering hedge fund - CNBC
4. Amazon Leo debuts new gigabit-speed 'Ultra' antenna, begins enterprise preview - About Amazon
5. Black Friday 2025 deals from Amazon, Walmart and Target are available now: The best sales we're shopping to kick off the holiday season - Yahoo
6. Futures Rise, Google Keeps Climbing; Novo Nordisk Dives - Investor's Business Daily
7. Microsoft and Nvidia Just Signed a Multibillion-Dollar Deal With Anthropic. Here's What It Really Means for Investors. - The Motley Fool
8. He told ChatGPT he was suicidal. It helped with his plan, family says. - USA Today
9. Google is crushing it. Why that’s worrying investors in Nvidia and other AI stocks. - MarketWatch
10. US banks scramble to assess da

In [107]:
%%writefile behavioral_alpha/ingestion/price_feed.py
import yfinance as yf

def get_price_history(ticker, period="1mo", interval="1d"):
    data = yf.download(ticker, period=period, interval=interval)
    return data


Overwriting behavioral_alpha/ingestion/price_feed.py


In [108]:
%%writefile behavioral_alpha/nlp/sentiment_engine.py
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

def get_sentiment(texts):
    scores = []
    for t in texts:
        inputs = tokenizer(t, return_tensors="pt", truncation=True)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=1).numpy()[0]
            score = probs[0]*(-1) + probs[2]*(1)
            scores.append(score)
    return np.mean(scores) if scores else 0


Overwriting behavioral_alpha/nlp/sentiment_engine.py


In [109]:
%%writefile behavioral_alpha/indicators/fgi.py
def calculate_fgi(sentiment_score, volatility=50, price_momentum=50, put_call_ratio=50, herding=50):
    components = [
        (sentiment_score + 1) * 50,  # sentiment [-1,1] -> [0,100]
        volatility,
        price_momentum,
        put_call_ratio,
        herding
    ]
    fgi = sum(components) / len(components)
    return fgi


Overwriting behavioral_alpha/indicators/fgi.py


In [110]:
%%writefile behavioral_alpha/signals/signal_engine.py
def generate_signal(fgi, oversold=20, overbought=80):
    if fgi < oversold:
        return "BUY"
    elif fgi > overbought:
        return "SELL"
    else:
        return "HOLD"



Overwriting behavioral_alpha/signals/signal_engine.py


In [111]:
%%writefile behavioral_alpha/database/db.py
import sqlite3

def init_db():
    conn = sqlite3.connect("behavioral_alpha.db")
    c = conn.cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS sentiment_data
                 (ticker TEXT, fgi REAL, signal TEXT)""")
    conn.commit()
    return conn




Overwriting behavioral_alpha/database/db.py


In [112]:
%%writefile behavioral_alpha/main.py
import sys
sys.path.append("behavioral_alpha")

from ingestion.news_feed import fetch_news
from ingestion.price_feed import get_price_history
from nlp.sentiment_engine import get_sentiment
from indicators.fgi import calculate_fgi
from signals.signal_engine import generate_signal
from database.db import init_db

ticker = "AAPL"  # Example stock

# 1. Fetch news
news = fetch_news(ticker)
print("Fetched news:", news)

# 2. Get historical prices
prices = get_price_history(ticker)
print("Recent closing prices:\n", prices['Close'].tail())

# 3. Get sentiment score
sent_score = get_sentiment(news)
print("Sentiment score:", sent_score)

# 4. Calculate FGI
fgi = calculate_fgi(sent_score)
print("Fear & Greed Index:", fgi)

# 5. Generate trade signal
signal = generate_signal(fgi)
print("Trade Signal:", signal)

# 6. Save to database
conn = init_db()
c = conn.cursor()
c.execute("INSERT INTO sentiment_data (ticker, fgi, signal) VALUES (?, ?, ?)",
          (ticker, fgi, signal))
conn.commit()
conn.close()


Overwriting behavioral_alpha/main.py


In [113]:
!python behavioral_alpha/main.py


2025-11-25 16:44:24.267006: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764089064.305924   47773 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764089064.314819   47773 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764089064.342420   47773 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764089064.342481   47773 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764089064.342486   47773 computation_placer.cc:177] computation placer alr

In [114]:
%%writefile behavioral_alpha/dashboard/app.py
import streamlit as st
import sqlite3
import pandas as pd

st.title("Behavioral Alpha Dashboard")

DB_PATH = "behavioral_alpha.db"

try:
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = c.fetchall()
    st.write("Tables in DB:", tables)

    query = "SELECT ticker, fgi, signal FROM sentiment_data ORDER BY rowid DESC LIMIT 10"
    df = pd.read_sql_query(query, conn)

    if df.empty:
        st.warning("No sentiment data found in the sentiment_data table.")
    else:
        st.subheader("Latest Sentiment Signals")
        st.table(df)

except Exception as e:
    st.error(f"Error loading dashboard: {e}")




Overwriting behavioral_alpha/dashboard/app.py


In [120]:
import sqlite3
from behavioral_alpha.ingestion.news_feed import fetch_news
from behavioral_alpha.nlp.sentiment_engine import get_sentiment
from behavioral_alpha.indicators.fgi import calculate_fgi
from behavioral_alpha.signals.signal_engine import generate_signal


conn = sqlite3.connect("behavioral_alpha.db")
c = conn.cursor()


c.execute("""
CREATE TABLE IF NOT EXISTS sentiment_data (
    ticker TEXT,
    fgi REAL,
    signal TEXT
)
""")
conn.commit()


tickers = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA', 'FB', 'NFLX', 'NVDA', 'AMD']

rows = []
for ticker in tickers:
    news = fetch_news()
    sentiment_score = get_sentiment(news)
    fgi = calculate_fgi(sentiment_score)
    signal = generate_signal(fgi)
    rows.append((ticker, fgi, signal))


c.executemany("INSERT INTO sentiment_data (ticker, fgi, signal) VALUES (?, ?, ?)", rows)
conn.commit()
conn.close()

print("Database populated with real stock data and signals!")


Database populated with real stock data and signals!


In [116]:
from pyngrok import ngrok
ngrok.set_auth_token("35yS1Azg3wlesaoQLJTym8LXp51_5wFx7ig9gTiSMny8G9YSt")


In [117]:
!nohup streamlit run behavioral_alpha/dashboard/app.py --server.port 8501 --server.address 0.0.0.0 &> /dev/null &


In [121]:
from pyngrok import ngrok

# Kill old tunnels (prevents errors)
ngrok.kill()

# Create a new public link for port 8501
public_url = ngrok.connect(8501)
public_url



<NgrokTunnel: "https://overbulkily-uncompulsory-mireille.ngrok-free.dev" -> "http://localhost:8501">